# 1. Ingest MEVA annotations and metadata

Fetch only the Kitware annotations and KRTD camera metadata from the MEVA repository. The temporary sparse clone is discarded; files are stored in `data/raw/`.

In [3]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import time

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
MEVA_REPOSITORY = "https://gitlab.kitware.com/meva/meva-data-repo.git"
KITWARE_PATH = Path("annotation/DIVA-phase-2/MEVA/kitware")
METADATA_PATH = Path("metadata/camera-models/krtd")

with tempfile.TemporaryDirectory(prefix="meva-") as temporary_directory:
    repository_dir = Path(temporary_directory) / "meva-data-repo"
    for attempt in range(1, 4):
        try:
            subprocess.run(
                [
                    "git", "-c", "http.version=HTTP/1.1", "clone", "--depth", "1",
                    "--filter=blob:none", "--sparse", MEVA_REPOSITORY, str(repository_dir),
                ],
                check=True,
            )
            subprocess.run(
                [
                    "git", "-c", "http.version=HTTP/1.1", "-C", str(repository_dir),
                    "sparse-checkout", "set", str(KITWARE_PATH), str(METADATA_PATH),
                ],
                check=True,
            )
            break
        except subprocess.CalledProcessError:
            shutil.rmtree(repository_dir, ignore_errors=True)
            if attempt == 3:
                raise
            time.sleep(attempt * 2)

    source_annotations = repository_dir / KITWARE_PATH
    source_metadata = repository_dir / METADATA_PATH
    target_annotations = DATA_DIR / "annotations" / "kitware"
    target_metadata = DATA_DIR / "metadata"
    shutil.rmtree(target_annotations, ignore_errors=True)
    shutil.rmtree(target_metadata, ignore_errors=True)
    target_annotations.mkdir(parents=True)
    for source_file in sorted(path for path in source_annotations.rglob("*") if path.is_file()):
        target_file = target_annotations / source_file.name
        if target_file.exists():
            raise FileExistsError(f"Cannot flatten Kitware files: {source_file.name}")
        shutil.copy2(source_file, target_file)
    shutil.copytree(source_metadata, target_metadata)

print(f"Staged files in {DATA_DIR}")

Cloning into '/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/meva-icp16za6/meva-data-repo'...
Updating files: 100% (2/2), done.


Staged files in /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw
